
# Pub/Sub Hands-On Lab — Google Cloud SDK (Python)
### GCP Training Program — Day 1, Module 1: Storage & Ingestion

## Problem Statement

A retail platform's order system needs to notify several independent downstream systems the
moment an order is placed — inventory needs to decrement stock, billing needs to charge the
customer, analytics needs to log the event, and a fraud-detection service needs to screen it —
without the order service knowing or caring who's listening, or waiting for any of them to finish.
This is exactly the problem **Pub/Sub** solves: a publisher fires one message once, and every
independent subscriber gets its own full copy, at its own pace, with delivery guarantees that
survive a subscriber being temporarily down.

This notebook builds up the full feature set needed to run that pattern reliably in production:

1. **"How do publishers and consumers agree on what a message even IS?"** — topics and schema
   validation *(Sections 4-5)*
2. **"How do multiple independent systems each get their own copy of every message?"** — fan-out
   subscriptions, pull vs. push *(Section 6)*
3. **"How does a message actually get published, including in-order when order matters?"** —
   publishing, batching, ordering keys *(Section 7)*
4. **"How does a consumer read messages, confirm it processed them, and what happens if it
   crashes mid-processing?"** — pull/ack/nack, streaming pull *(Sections 8-9)*
5. **"What happens to a message that a subscriber can never successfully process?"** —
   dead-letter topics *(Section 10)*
6. **"Can we replay messages we already consumed — say, after finding a bug in our processing
   code?"** — snapshots and seek-by-timestamp *(Section 11)*
7. **"How do we protect a slow consumer from being overwhelmed, and control retry timing?"** —
   flow control and retry policies *(Section 12)*
8. **"Can a subscriber receive only the messages relevant to it, without filtering in application
   code?"** — server-side message filtering *(Section 13)*
9. **"Can we get a stronger guarantee than 'might occasionally redeliver'?"** — exactly-once
   delivery *(Section 14)*

**Authentication only** uses the `gcloud` CLI/Cloud SDK (via Colab's built-in auth flow) — every
Pub/Sub operation itself goes through the **Python client library** (`google-cloud-pubsub`), not
the CLI. Every operation includes a **🖥️ Check it in the console** pointer showing where the same
thing is visible in the Cloud Console.

**This notebook is fully self-contained.** Just run the cells top to bottom:
1. The **Authenticate** cell will pop up a Google sign-in flow.
2. The **Configure** cell will *ask you* for your Project ID — topic/subscription names are
   generated automatically with a timestamp suffix so nothing collides with a previous run.
3. Everything after that runs against your own project automatically.

**Prerequisites (mention to the class before running):**
1. A GCP project with billing enabled and the **Pub/Sub API** enabled
   (`gcloud services enable pubsub.googleapis.com` — a cell below does this for you).
2. IAM role on the project: `roles/pubsub.editor` (or equivalent) for the account you sign in with.
3. Python package `google-cloud-pubsub` (installed in a cell below).

**How to run in class:** execute cells top-to-bottom. Later sections (replay, dead-letter,
ordering) assume the topic and subscriptions created early in the notebook still exist.


## 1. Authenticate This Session
Sign in with the Google account that has access to your GCP project. This is the only place in
the notebook that leans on the Cloud SDK rather than the Python client library.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated in Colab. This also configures the gcloud CLI for this session.")
else:
    print("Not running in Colab — assuming gcloud Application Default Credentials are set up.")
    print("If this is your first time, run in a terminal:")
    print("  gcloud auth login")
    print("  gcloud auth application-default login")


> 🖥️ **Check it in the console:** **IAM & Admin > IAM** → confirm the signed-in account shows a
> role covering Pub/Sub (e.g. **Pub/Sub Editor** or **Owner/Editor**) — this is the identity every
> later console checkpoint in this notebook will be checking as.


## 2. Install & Import the Client Library

In [ ]:
!pip install --quiet "google-cloud-pubsub>=2.18.0"

In [ ]:
import time
import json
from google.cloud import pubsub_v1
from google.pubsub_v1.types import Schema, Encoding
from google.api_core import exceptions as gcs_exceptions
from google.api_core.retry import Retry
from google.protobuf import duration_pb2

print("google-cloud-pubsub imported successfully")

## 3. Configure Your Project
Enter your Project ID when prompted. Topic, subscription, schema, and snapshot names are all
generated with a shared timestamp suffix so re-running this notebook never collides with a
previous run — nothing else to hand-edit.

In [ ]:
PROJECT_ID = input("Enter your GCP Project ID: ").strip()

!gcloud config set project {PROJECT_ID}
!gcloud services enable pubsub.googleapis.com cloudfunctions.googleapis.com \
    cloudbuild.googleapis.com run.googleapis.com artifactregistry.googleapis.com \
    --project={PROJECT_ID}

_suffix = int(time.time())
TOPIC_ID              = f"training-topic-{_suffix}"
SCHEMA_TOPIC_ID        = f"training-schema-topic-{_suffix}"
SCHEMA_ID              = f"training-schema-{_suffix}"
SUB_A_ID               = f"training-sub-a-{_suffix}"
SUB_B_ID               = f"training-sub-b-{_suffix}"
ORDERED_SUB_ID         = f"training-sub-ordered-{_suffix}"
FILTERED_SUB_ID        = f"training-sub-filtered-{_suffix}"
EXACTLY_ONCE_SUB_ID    = f"training-sub-exactly-once-{_suffix}"
DLQ_TOPIC_ID           = f"training-dlq-topic-{_suffix}"
DLQ_SOURCE_SUB_ID      = f"training-sub-with-dlq-{_suffix}"
SNAPSHOT_ID            = f"training-snapshot-{_suffix}"

publisher = pubsub_v1.PublisherClient()
subscriber = pubsub_v1.SubscriberClient()

print("Project:", PROJECT_ID)
print("Main topic will be:", TOPIC_ID)

## 4. Topics
### 4.1 Create a Topic

In [ ]:
topic_path = publisher.topic_path(PROJECT_ID, TOPIC_ID)
topic = publisher.create_topic(request={"name": topic_path})
print("Created topic:", topic.name)


> 🖥️ **Check it in the console:** **Pub/Sub > Topics** → your new topic should now be listed —
> this is the central hub every subscription and every publish call in this notebook attaches to.


### 4.2 List All Topics in the Project

In [ ]:
project_path = f"projects/{PROJECT_ID}"
for t in publisher.list_topics(request={"project": project_path}):
    print(" -", t.name)


> 🖥️ **Check it in the console:** same **Pub/Sub > Topics** page → every topic printed above
> should appear in this list, project-wide.


### 4.3 Get Topic Details

In [ ]:
fetched = publisher.get_topic(request={"topic": topic_path})
print("Name:  ", fetched.name)
print("Labels:", dict(fetched.labels))


> 🖥️ **Check it in the console:** **Pub/Sub > Topics > [your topic]** → the topic's overview page
> shows the same name and labels just printed, plus a **Metrics** tab (publish/message-count
> charts) that becomes genuinely useful once messages start flowing in Section 7.


## 5. Schema Registry — Contract-Driven Ingestion
### 5.1 Define & Register an Avro Schema
Schema Registry lets a topic reject messages that don't match an agreed shape — useful for
enforcing a data contract between publishers and consumers instead of discovering a malformed
payload downstream in BigQuery.

**Caveat for the class:** Pub/Sub schema support (Avro/Protocol Buffer, JSON vs BINARY encoding)
has evolved across SDK versions — if a call below doesn't match what you see in the docs, check
your installed `google-cloud-pubsub` version first.

In [ ]:
schema_client = pubsub_v1.SchemaServiceClient()
schema_path = schema_client.schema_path(PROJECT_ID, SCHEMA_ID)

avro_definition = json.dumps({
    "type": "record",
    "name": "TrainingEvent",
    "fields": [
        {"name": "event_id", "type": "string"},
        {"name": "event_type", "type": "string"},
        {"name": "amount", "type": "double"},
    ],
})

schema = schema_client.create_schema(
    request={
        "parent": f"projects/{PROJECT_ID}",
        "schema": {
            "name": schema_path,
            "type_": Schema.Type.AVRO,
            "definition": avro_definition,
        },
        "schema_id": SCHEMA_ID,
    }
)
print("Created schema:", schema.name)


> 🖥️ **Check it in the console:** **Pub/Sub > Schemas** → your new schema should be listed, with
> its Avro definition viewable directly — this is the data contract Section 5.2's topic will now
> enforce on every publish.


### 5.2 Create a Topic That Enforces the Schema

In [ ]:
schema_topic_path = publisher.topic_path(PROJECT_ID, SCHEMA_TOPIC_ID)

schema_topic = publisher.create_topic(
    request={
        "name": schema_topic_path,
        "schema_settings": {
            "schema": schema_path,
            "encoding": Encoding.JSON,
        },
    }
)
print("Created schema-enforced topic:", schema_topic.name)


> 🖥️ **Check it in the console:** **Pub/Sub > Topics > [schema topic]** → the **Schema** tab shows
> the attached schema and encoding (JSON) — confirming enforcement is wired up before you test it
> in the next two cells.


### 5.3 Publish a Valid Message (Passes Schema Validation)

In [ ]:
valid_payload = json.dumps({"event_id": "evt-001", "event_type": "purchase", "amount": 42.5}).encode("utf-8")
future = publisher.publish(schema_topic_path, data=valid_payload)
print("Published valid message, message ID:", future.result())


> 🖥️ **Check it in the console:** **Pub/Sub > Topics > [schema topic] > Metrics** tab → the
> publish-request count should tick up by one — the console's confirmation that a schema-valid
> message actually went through.


### 5.4 Publish an Invalid Message (Fails Schema Validation)
**Teaching point:** the server rejects this before it ever reaches a subscriber — the contract is
enforced at publish time, not left for downstream consumers to discover.

In [ ]:
invalid_payload = json.dumps({"event_id": "evt-002", "amount": "not-a-number"}).encode("utf-8")

try:
    future = publisher.publish(schema_topic_path, data=invalid_payload)
    future.result()
    print("Unexpectedly succeeded — check whether schema enforcement is active.")
except gcs_exceptions.GoogleAPICallError as e:
    print("Rejected by schema validation, as expected:", str(e)[:200])


## Pub/Sub Architecture Overview

Before creating any subscriptions, it's worth laying out how the pieces actually fit together —
everything below is one architecture, not several unrelated features.

```
                                   ┌─────────────────┐
                                   │   Subscription A │──── pull() ────▶ Consumer A
                                   │   (independent    │                (polls when ready)
                              ┌───▶│    ack state)     │
                              │    └─────────────────┘
   Publisher ──publish()──▶ Topic
                              │    ┌─────────────────┐
                              └───▶│   Subscription B │──push──▶ HTTPS endpoint
                                   │   (independent    │        (Consumer B, called
                                   │    ack state)     │         by Pub/Sub itself)
                                   └─────────────────┘
```

**The core guarantees this architecture provides:**

- **Fan-out is automatic and free.** Every subscription attached to a topic gets its own complete,
  independent copy of every message — adding Subscription B doesn't cost Subscription A anything,
  and acking on one has zero effect on the other's copy (demonstrated concretely in Section 8.4).
- **Delivery is at-least-once by default**, not exactly-once (Section 14 covers the newer
  exactly-once option). A message might rarely be redelivered even after a successful ack — design
  consumers to be idempotent where that matters.
- **The ack deadline is the whole redelivery mechanism.** A pulled-but-unacked message becomes
  eligible for redelivery once its ack deadline passes (or immediately, via an explicit nack —
  Section 8.3). There's no separate "retry" system bolted on; deadline expiry *is* the retry
  trigger, which is also what the retry *policy* in Section 12.2 tunes the timing of.

**Pull vs. push — the same subscription concept, two different delivery mechanics:**

| | Pull | Push |
|---|---|---|
| **Who initiates delivery** | Your consumer code calls `subscriber.pull()` (or opens a streaming pull) | Pub/Sub itself calls an HTTPS endpoint you own |
| **Best for** | Long-running consumers, batch processing, full control over pacing | Lightweight, event-driven compute (Cloud Functions, Cloud Run) that shouldn't have to run a polling loop |
| **Ack mechanism** | Your code explicitly acks/nacks | Endpoint's HTTP response code — 2xx acks, anything else (or a timeout) is treated as a nack |
| **Infrastructure needed** | Just your consumer process | A real, reachable HTTPS endpoint, which Pub/Sub must be authorized to invoke |

Section 6.1 demonstrates **pull** with two independent subscribers (fan-out). Section 6.2
demonstrates **push** for real, by deploying an actual Cloud Function as the push target — so you
see a genuine push delivery happen, not just the configuration shape.


## 6. Subscriptions — Pull, Push & Fan-Out
### 6.1 Create Two Independent Pull Subscriptions on the Same Topic
This is the fan-out pattern: every subscription attached to a topic gets **its own copy** of every
message. Subscriber A and Subscriber B below are fully independent — acking a message on A has no
effect on B's copy.

In [ ]:
sub_a_path = subscriber.subscription_path(PROJECT_ID, SUB_A_ID)
sub_b_path = subscriber.subscription_path(PROJECT_ID, SUB_B_ID)

subscriber.create_subscription(request={"name": sub_a_path, "topic": topic_path})
subscriber.create_subscription(request={"name": sub_b_path, "topic": topic_path})

print("Created subscriber A:", sub_a_path)
print("Created subscriber B:", sub_b_path)


> 🖥️ **Check it in the console:** **Pub/Sub > Subscriptions** → both `training-sub-a-...` and
> `training-sub-b-...` should be listed, each showing the same parent topic — this is the fan-out
> relationship: one topic, two fully independent subscriptions.



### 6.2 Create a Push Subscription — With a Real, Working Endpoint

**Why this is different from a typical demo:** push subscriptions are usually shown as
configuration syntax only, since they need a real HTTPS endpoint — which most training notebooks
don't have handy. Here, we deploy an actual minimal **Cloud Function (2nd gen)** as that endpoint,
so push delivery genuinely happens and you can see it land in Cloud Logging, not just trust that
the configuration *would* work.

**Why Cloud Functions specifically:** Pub/Sub automatically trusts `*.cloudfunctions.net` /
`*.run.app` URLs for push delivery — no domain-ownership verification step required, unlike an
arbitrary external HTTPS endpoint. This is the path of least friction for a live class demo.

**Cost & time note:** this deploys real (tiny, free-tier-eligible) infrastructure and takes
**2-4 minutes** to build and deploy — noticeably slower than any other cell so far in this
notebook, since it involves a container build behind the scenes, not just an API call.


In [ ]:
import os

os.makedirs("push_function", exist_ok=True)

with open("push_function/main.py", "w") as f:
    f.write("""
import base64
import functions_framework

@functions_framework.http
def pubsub_push_handler(request):
    envelope = request.get_json()
    if not envelope or "message" not in envelope:
        return ("Bad Request: no Pub/Sub message received", 400)

    message = envelope["message"]
    data = base64.b64decode(message.get("data", "")).decode("utf-8") if message.get("data") else ""
    attributes = message.get("attributes", {})

    print(f"Received push message: data={data!r} attributes={attributes}")
    return ("", 204)
""")

with open("push_function/requirements.txt", "w") as f:
    f.write("functions-framework==3.*\n")

print("Wrote push_function/main.py and requirements.txt")



Now deploy it. `--trigger-http` with `--no-allow-unauthenticated` keeps the function private —
Pub/Sub will authenticate to it using a service account and an OIDC identity token, the
production-correct way to secure a push endpoint (rather than leaving it open to the whole
internet).


In [ ]:
PUSH_FUNCTION_NAME = f"training-push-handler-{_suffix}"

!gcloud functions deploy {PUSH_FUNCTION_NAME} \
    --gen2 \
    --runtime=python312 \
    --region={REGION if 'REGION' in dir() else 'us-central1'} \
    --source=push_function \
    --entry-point=pubsub_push_handler \
    --trigger-http \
    --no-allow-unauthenticated \
    --project={PROJECT_ID} \
    --quiet

push_endpoint_url = !gcloud functions describe {PUSH_FUNCTION_NAME} \
    --gen2 --region={REGION if 'REGION' in dir() else 'us-central1'} \
    --project={PROJECT_ID} --format="value(serviceConfig.uri)"
PUSH_ENDPOINT_URL = push_endpoint_url[0].strip()
print("Deployed push endpoint:", PUSH_ENDPOINT_URL)



Create a dedicated service account for Pub/Sub to authenticate as, grant it permission to invoke
the function, then create the push subscription pointing at it with that identity attached.


In [ ]:
PUSH_SA_NAME = f"training-push-invoker-{_suffix}"
PUSH_SA_EMAIL = f"{PUSH_SA_NAME}@{PROJECT_ID}.iam.gserviceaccount.com"

!gcloud iam service-accounts create {PUSH_SA_NAME} \
    --display-name="Pub/Sub push invoker for training lab" \
    --project={PROJECT_ID} --quiet

!gcloud functions add-invoker-policy-binding {PUSH_FUNCTION_NAME} \
    --gen2 --region={REGION if 'REGION' in dir() else 'us-central1'} \
    --member="serviceAccount:{PUSH_SA_EMAIL}" \
    --project={PROJECT_ID} --quiet

push_sub_id = f"training-sub-push-{_suffix}"
push_sub_path = subscriber.subscription_path(PROJECT_ID, push_sub_id)

subscriber.create_subscription(
    request={
        "name": push_sub_path,
        "topic": topic_path,
        "push_config": {
            "push_endpoint": PUSH_ENDPOINT_URL,
            "oidc_token": {"service_account_email": PUSH_SA_EMAIL},
        },
    }
)
print("Created push subscription with authenticated push to:", PUSH_ENDPOINT_URL)



### 6.3 Publish a Message and Watch It Get Pushed for Real
IAM changes can take a few seconds to propagate — if the first attempt shows an auth error in the
logs, just re-run the publish cell once.


In [ ]:
import time

publisher.publish(topic_path, data=b"pushed via a real Cloud Function endpoint!", demo="push").result()
print("Published. Waiting 15 seconds for push delivery + log propagation...")
time.sleep(15)

!gcloud functions logs read {PUSH_FUNCTION_NAME} --gen2 \
    --region={REGION if 'REGION' in dir() else 'us-central1'} \
    --project={PROJECT_ID} --limit=10



**What to look for:** the log output above should include a line like
`Received push message: data='pushed via a real Cloud Function endpoint!' attributes={'demo': 'push'}`
— proof Pub/Sub itself called this endpoint and delivered the message, not something simulated
locally in this notebook.

> 🖥️ **Check it in the console:** **Pub/Sub > Subscriptions > [push subscription]** → **Delivery
> type** shows **Push**, with the Cloud Function's URL and the attached service account identity.
> **Cloud Run > [the same function, since gen2 functions run on Cloud Run]** → the **Logs** tab
> shows the same received-message log line, in the console instead of the CLI. **Cloud Functions
> > [function name] > Metrics** tab shows an invocation count of at least 1.


### 6.3 List All Subscriptions on the Topic

In [ ]:
for s in publisher.list_topic_subscriptions(request={"topic": topic_path}):
    print(" -", s)


> 🖥️ **Check it in the console:** **Pub/Sub > Topics > [your topic] > Subscriptions** tab → every
> subscription attached to this topic is listed here in one place, the same list this cell just
> printed via the API.


## 7. Publishing Messages
### 7.1 Publish a Single Message With Attributes

In [ ]:
future = publisher.publish(
    topic_path,
    data=b"Hello from the Pub/Sub hands-on lab!",
    source="training-notebook",
    priority="normal",
)
print("Published message ID:", future.result())


> 🖥️ **Check it in the console:** **Pub/Sub > Topics > [your topic] > Metrics** tab → the publish
> count increments — this is the simplest possible way to confirm a message actually left the
> publisher, without needing to pull it on the subscriber side at all.


### 7.2 Publish a Batch of Messages Asynchronously
`publish()` returns a future immediately — the client batches messages under the hood and sends
them together, so firing off many publishes in a loop is efficient without any extra code.

In [ ]:
futures = []
for i in range(10):
    f = publisher.publish(topic_path, data=f"batch message {i}".encode("utf-8"), index=str(i))
    futures.append(f)

message_ids = [f.result() for f in futures]
print(f"Published {len(message_ids)} messages, e.g. first ID:", message_ids[0])


> 🖥️ **Check it in the console:** same **Metrics** tab → the publish count should jump by 10 at
> once — worth pointing out the chart doesn't distinguish "10 individual calls" from "1 batched
> call," since batching is a client-side efficiency, not a different wire format.


### 7.3 Ordered Delivery With an Ordering Key
Messages sharing the same `ordering_key` are delivered to a subscriber in the order they were
published — normally Pub/Sub gives no ordering guarantee at all. Requires ordering enabled on
both the publisher client and the subscription.

In [ ]:
ordered_publisher = pubsub_v1.PublisherClient(
    publisher_options=pubsub_v1.types.PublisherOptions(enable_message_ordering=True)
)

ordered_sub_path = subscriber.subscription_path(PROJECT_ID, ORDERED_SUB_ID)
subscriber.create_subscription(
    request={"name": ordered_sub_path, "topic": topic_path, "enable_message_ordering": True}
)

for i in range(5):
    ordered_publisher.publish(topic_path, data=f"ordered event {i}".encode("utf-8"), ordering_key="order-key-1")

print("Published 5 messages under ordering_key='order-key-1' to", ordered_sub_path)


> 🖥️ **Check it in the console:** **Pub/Sub > Subscriptions > [ordered subscription]** → the
> **Message ordering** field should read **Enabled** — required on both this subscription and the
> publisher client for the ordering guarantee to actually hold.


## 8. Synchronous Pull & Acknowledge
### 8.1 Pull a Batch of Messages From Subscriber A

In [ ]:
response = subscriber.pull(
    request={"subscription": sub_a_path, "max_messages": 10},
    retry=Retry(deadline=30),
)

print(f"Pulled {len(response.received_messages)} message(s) on Subscriber A:")
for rm in response.received_messages:
    print(" -", rm.message.data[:60], "| attributes:", dict(rm.message.attributes))


> 🖥️ **Check it in the console:** **Pub/Sub > Subscriptions > [Subscriber A] > Metrics** tab →
> **Unacked message count** reflects exactly what this pull just retrieved, before the next
> cell acks it.


### 8.2 Acknowledge the Pulled Messages
Once acked, these messages will **not** be redelivered to Subscriber A. Subscriber B (Section 8.4)
still has its own independent, unacked copies.

In [ ]:
ack_ids = [rm.ack_id for rm in response.received_messages]
if ack_ids:
    subscriber.acknowledge(request={"subscription": sub_a_path, "ack_ids": ack_ids})
    print(f"Acknowledged {len(ack_ids)} message(s) on Subscriber A.")
else:
    print("No messages were pulled — re-run Section 7.1/7.2 to publish more first.")


> 🖥️ **Check it in the console:** same subscription **Metrics** tab → **Unacked message count**
> should drop to 0 (or by the number acked) — the console's confirmation that acking really did
> register server-side, not just locally in this notebook's variables.


### 8.3 Negative-Acknowledge (nack) — Force Immediate Redelivery
Publish one more message, pull it, then **nack** it instead of acking — this resets its ack
deadline to 0, making it immediately eligible for redelivery instead of waiting out the full ack
deadline.

In [ ]:
publisher.publish(topic_path, data=b"message to be nacked").result()
time.sleep(2)  # brief pause so the message is available to pull

response = subscriber.pull(request={"subscription": sub_a_path, "max_messages": 1})
if response.received_messages:
    nack_id = response.received_messages[0].ack_id
    subscriber.modify_ack_deadline(
        request={"subscription": sub_a_path, "ack_ids": [nack_id], "ack_deadline_seconds": 0}
    )
    print("Nacked one message — it will be redelivered immediately instead of after the ack deadline.")
else:
    print("Nothing pulled — the nack-demo message may already have been consumed by an earlier pull.")

### 8.4 Confirm Fan-Out: Subscriber B Still Has Its Own Full Backlog
Subscriber B never had anything acked on it — pulling here shows every message published earlier in
Section 7 is still waiting for it, completely independent of what happened on Subscriber A above.

In [ ]:
response_b = subscriber.pull(request={"subscription": sub_b_path, "max_messages": 20})
print(f"Subscriber B has {len(response_b.received_messages)} message(s) still pending — independent of Subscriber A.")

ack_ids_b = [rm.ack_id for rm in response_b.received_messages]
if ack_ids_b:
    subscriber.acknowledge(request={"subscription": sub_b_path, "ack_ids": ack_ids_b})
    print(f"Acknowledged {len(ack_ids_b)} message(s) on Subscriber B.")


> 🖥️ **Check it in the console:** **Pub/Sub > Subscriptions > [Subscriber B] > Metrics** tab →
> before this cell runs, **Unacked message count** should already show the full backlog — visible,
> concrete proof of fan-out: Subscriber B never lost anything just because Subscriber A acked its
> own copy.


## 9. Streaming Pull — Asynchronous Callback-Based Consumer
This is the pattern most production consumers use instead of polling with `pull()` in a loop: the
client opens a long-lived stream and invokes your callback as messages arrive. This cell runs for
~15 seconds, publishing a few messages partway through, then shuts the stream down cleanly — long
enough to demo live without blocking the rest of the notebook.

In [ ]:
received_via_streaming = []

def callback(message):
    received_via_streaming.append(message.data)
    print("Streaming pull received:", message.data[:60])
    message.ack()

streaming_future = subscriber.subscribe(sub_a_path, callback=callback)
print("Listening on Subscriber A for 15 seconds...")

time.sleep(3)
for i in range(3):
    publisher.publish(topic_path, data=f"streamed message {i}".encode("utf-8")).result()

time.sleep(12)
streaming_future.cancel()
try:
    streaming_future.result(timeout=5)
except Exception:
    pass

print(f"\nStreaming pull received {len(received_via_streaming)} message(s) total.")


> 🖥️ **Check it in the console:** **Pub/Sub > Subscriptions > [Subscriber A] > Metrics** tab,
> watched live while this cell runs → **Unacked message count** should briefly rise as the 3
> streamed messages are published, then drop back to 0 as the callback acks each one — the
> clearest way to see streaming pull's behavior in real time rather than after the fact.


## 10. Dead-Letter Topics
### 10.1 Create the Dead-Letter Topic and Grant Pub/Sub Permission to Publish to It
The Pub/Sub service agent needs explicit IAM permission to publish failed messages to your
dead-letter topic, and to forward from the source subscription — this step is easy to forget and
a common reason dead-lettering silently doesn't work.

In [ ]:
dlq_topic_path = publisher.topic_path(PROJECT_ID, DLQ_TOPIC_ID)
publisher.create_topic(request={"name": dlq_topic_path})
print("Created dead-letter topic:", dlq_topic_path)

# Fetch the project number (needed to build the Pub/Sub service agent email)
project_number = !gcloud projects describe {PROJECT_ID} --format="value(projectNumber)"
project_number = project_number[0].strip()
pubsub_service_agent = f"serviceAccount:service-{project_number}@gcp-sa-pubsub.iam.gserviceaccount.com"
print("Pub/Sub service agent:", pubsub_service_agent)

In [ ]:
!gcloud pubsub topics add-iam-policy-binding {DLQ_TOPIC_ID} \
    --project={PROJECT_ID} \
    --member="{pubsub_service_agent}" \
    --role="roles/pubsub.publisher"


> 🖥️ **Check it in the console:** **Pub/Sub > Topics > [dead-letter topic] > Permissions** tab →
> the Pub/Sub service agent should now be listed with the **Pub/Sub Publisher** role — the exact
> permission this `gcloud` command just granted, visible in the console's IAM UI.


### 10.2 Create a Subscription With a Dead-Letter Policy
After `max_delivery_attempts` failed delivery attempts (i.e. the subscriber keeps nacking or
letting the ack deadline expire), Pub/Sub forwards the message to the dead-letter topic instead of
retrying forever.

In [ ]:
dlq_source_sub_path = subscriber.subscription_path(PROJECT_ID, DLQ_SOURCE_SUB_ID)

subscriber.create_subscription(
    request={
        "name": dlq_source_sub_path,
        "topic": topic_path,
        "dead_letter_policy": {
            "dead_letter_topic": dlq_topic_path,
            "max_delivery_attempts": 5,
        },
    }
)
print("Created subscription with dead-letter policy:", dlq_source_sub_path)

# The source subscription also needs the service agent to have subscribe permission,
# so it can pull-and-forward on your behalf:
!gcloud pubsub subscriptions add-iam-policy-binding {DLQ_SOURCE_SUB_ID} \
    --project={PROJECT_ID} \
    --member="{pubsub_service_agent}" \
    --role="roles/pubsub.subscriber"


> 🖥️ **Check it in the console:** **Pub/Sub > Subscriptions > [DLQ source subscription]** →
> the **Dead lettering** field shows the configured dead-letter topic and
> **Maximum delivery attempts: 5** — confirming the policy from this cell landed correctly before
> the next cell tries to trigger it.


### 10.3 Force a Message to Dead-Letter by Repeatedly Nacking It
**Time note:** this loop nacks the same message up to `max_delivery_attempts` times, waiting
briefly between attempts for Pub/Sub to redeliver it. Expect this cell to take a minute or two.

In [ ]:
publisher.publish(topic_path, data=b"this message will be dead-lettered").result()

for attempt in range(6):
    time.sleep(5)
    resp = subscriber.pull(request={"subscription": dlq_source_sub_path, "max_messages": 1})
    if not resp.received_messages:
        print(f"Attempt {attempt}: nothing pulled (already dead-lettered, or still waiting on redelivery).")
        continue
    ack_id = resp.received_messages[0].ack_id
    subscriber.modify_ack_deadline(
        request={"subscription": dlq_source_sub_path, "ack_ids": [ack_id], "ack_deadline_seconds": 0}
    )
    print(f"Attempt {attempt}: pulled and nacked again.")

dlq_check = subscriber.subscription_path(PROJECT_ID, f"{DLQ_SOURCE_SUB_ID}-dlq-check")
subscriber.create_subscription(request={"name": dlq_check, "topic": dlq_topic_path})
dlq_resp = subscriber.pull(request={"subscription": dlq_check, "max_messages": 5})
print(f"\nMessages on the dead-letter topic: {len(dlq_resp.received_messages)}")
for rm in dlq_resp.received_messages:
    print(" -", rm.message.data)
    subscriber.acknowledge(request={"subscription": dlq_check, "ack_ids": [rm.ack_id]})


> 🖥️ **Check it in the console:** **Pub/Sub > Topics > [dead-letter topic] > Metrics** tab →
> after the nacking loop finishes, the publish count here should show 1 — the message Pub/Sub
> itself forwarded after exhausting delivery attempts, not something this notebook published
> directly.


## 11. Snapshots & Replay
### 11.1 Take a Snapshot of Subscriber A's Current Backlog Position
A snapshot freezes a subscription's acknowledgment state at a point in time — seeking back to it
later **replays** every message that was unacked (or published after) at snapshot time.

In [ ]:
snapshot_path = subscriber.snapshot_path(PROJECT_ID, SNAPSHOT_ID)
subscriber.create_snapshot(request={"name": snapshot_path, "subscription": sub_a_path})
print("Created snapshot:", snapshot_path)


> 🖥️ **Check it in the console:** **Pub/Sub > Snapshots** → your new snapshot should be listed,
> showing which subscription it was taken from and its expiration date (snapshots auto-expire
> after 7 days by default if unused).


### 11.2 Publish New Messages, Then Replay Back to the Snapshot
Publish a few more messages, drain them normally, then **seek** the subscription back to the
snapshot — every message that existed at snapshot time becomes redeliverable again, demonstrating
replay.

In [ ]:
for i in range(3):
    publisher.publish(topic_path, data=f"post-snapshot message {i}".encode("utf-8")).result()
time.sleep(2)

# Drain and ack everything currently pending, so the "replay" below is unambiguous
drain = subscriber.pull(request={"subscription": sub_a_path, "max_messages": 50})
if drain.received_messages:
    subscriber.acknowledge(
        request={"subscription": sub_a_path, "ack_ids": [rm.ack_id for rm in drain.received_messages]}
    )
print(f"Drained and acked {len(drain.received_messages)} message(s) before replay.")

subscriber.seek(request={"subscription": sub_a_path, "snapshot": snapshot_path})
print("Sought Subscriber A back to the snapshot.")

replayed = subscriber.pull(request={"subscription": sub_a_path, "max_messages": 50})
print(f"Messages redelivered after replay: {len(replayed.received_messages)}")
if replayed.received_messages:
    subscriber.acknowledge(
        request={"subscription": sub_a_path, "ack_ids": [rm.ack_id for rm in replayed.received_messages]}
    )

### 11.3 Replay by Timestamp Instead of a Snapshot
Seeking also accepts a timestamp directly — useful for "replay everything published in the last
10 minutes" without having created a snapshot in advance.

In [ ]:
from google.protobuf.timestamp_pb2 import Timestamp

ten_minutes_ago = Timestamp()
ten_minutes_ago.FromSeconds(int(time.time()) - 600)

subscriber.seek(request={"subscription": sub_a_path, "time": ten_minutes_ago})
print("Sought Subscriber A back to 10 minutes ago — messages from that window become redeliverable.")


> 🖥️ **Check it in the console:** **Pub/Sub > Subscriptions > [Subscriber A] > Metrics** tab →
> **Unacked message count** should rise again after this seek — the same "replay" effect as the
> snapshot-based seek in 11.2, just targeting a point in time instead of a saved snapshot.


## 12. Flow Control & Retry Policy
### 12.1 Client-Side Flow Control
Flow control caps how many messages/bytes the streaming pull client will hold in memory
un-acked at once — protects a slow consumer from being overwhelmed by a fast publisher.

In [ ]:
flow_control = pubsub_v1.types.FlowControl(max_messages=5, max_bytes=1024 * 1024)

limited = []
def limited_callback(message):
    limited.append(message.data)
    time.sleep(1)  # simulate slow processing
    message.ack()

for i in range(8):
    publisher.publish(topic_path, data=f"flow-control message {i}".encode("utf-8")).result()

flow_future = subscriber.subscribe(sub_b_path, callback=limited_callback, flow_control=flow_control)
time.sleep(8)
flow_future.cancel()
try:
    flow_future.result(timeout=5)
except Exception:
    pass

print(f"Processed {len(limited)} message(s) under flow control (max_messages=5 in flight at a time).")

### 12.2 Server-Side Retry Policy
Controls how long Pub/Sub waits before redelivering an unacked message, with exponential backoff
between `minimum_backoff` and `maximum_backoff` — set this on the subscription itself, no client
code required to benefit from it.

In [ ]:
retry_sub_id = f"training-sub-retry-{_suffix}"
retry_sub_path = subscriber.subscription_path(PROJECT_ID, retry_sub_id)

subscriber.create_subscription(
    request={
        "name": retry_sub_path,
        "topic": topic_path,
        "retry_policy": {
            "minimum_backoff": duration_pb2.Duration(seconds=10),
            "maximum_backoff": duration_pb2.Duration(seconds=600),
        },
    }
)
print("Created subscription with a custom retry (backoff) policy:", retry_sub_path)


> 🖥️ **Check it in the console:** **Pub/Sub > Subscriptions > [retry-policy subscription]** →
> the **Retry policy** field shows **Minimum backoff: 10s, Maximum backoff: 600s** — enforced
> entirely server-side, no client code required to benefit from it.


## 13. Message Filtering
A subscription can filter on message attributes server-side — only matching messages are ever
delivered to it, the rest are auto-acked without your code ever seeing them. Cheaper and simpler
than pulling everything and filtering in application code.

In [ ]:
filtered_sub_path = subscriber.subscription_path(PROJECT_ID, FILTERED_SUB_ID)

subscriber.create_subscription(
    request={
        "name": filtered_sub_path,
        "topic": topic_path,
        "filter": 'attributes.priority = "high"',
    }
)

publisher.publish(topic_path, data=b"low priority event", priority="low").result()
publisher.publish(topic_path, data=b"high priority event", priority="high").result()
time.sleep(2)

resp = subscriber.pull(request={"subscription": filtered_sub_path, "max_messages": 10})
print(f"Filtered subscription received {len(resp.received_messages)} message(s) (expect only the high-priority one):")
for rm in resp.received_messages:
    print(" -", rm.message.data, "| priority:", rm.message.attributes.get("priority"))

if resp.received_messages:
    subscriber.acknowledge(
        request={"subscription": filtered_sub_path, "ack_ids": [rm.ack_id for rm in resp.received_messages]}
    )


> 🖥️ **Check it in the console:** **Pub/Sub > Subscriptions > [filtered subscription]** → the
> **Filter** field shows `attributes.priority = "high"` — and its **Metrics** tab's
> **Unacked message count** should reflect only the one high-priority message, since the low-
> priority one was auto-acknowledged by Pub/Sub without ever reaching this notebook's pull call.


## 14. Exactly-Once Delivery
By default Pub/Sub gives **at-least-once** delivery (a message might be redelivered after you've
already acked it, in rare cases). Enabling exactly-once delivery on a subscription lets you check
whether an ack actually succeeded — if it did, that message is guaranteed not to be redelivered.

**Caveat for the class:** exactly-once delivery is a newer subscription feature with its own
throughput/latency trade-offs — check current GCP documentation before recommending it for a
high-throughput production pipeline.

In [ ]:
exactly_once_sub_path = subscriber.subscription_path(PROJECT_ID, EXACTLY_ONCE_SUB_ID)

subscriber.create_subscription(
    request={
        "name": exactly_once_sub_path,
        "topic": topic_path,
        "enable_exactly_once_delivery": True,
    }
)

publisher.publish(topic_path, data=b"exactly-once demo message").result()
time.sleep(2)

resp = subscriber.pull(request={"subscription": exactly_once_sub_path, "max_messages": 1})
if resp.received_messages:
    ack_id = resp.received_messages[0].ack_id
    ack_response = subscriber.acknowledge(
        request={"subscription": exactly_once_sub_path, "ack_ids": [ack_id]}
    )
    print("Ack sent. With exactly-once delivery enabled, a successful ack guarantees no redelivery.")
else:
    print("Nothing pulled yet — re-run this cell if the publish hasn't propagated.")


> 🖥️ **Check it in the console:** **Pub/Sub > Subscriptions > [exactly-once subscription]** →
> the **Exactly-once delivery** field should read **Enabled** — the setting that makes a
> successful ack response an actual guarantee rather than a best-effort signal.


## 15. Cleanup — Delete Subscriptions, Snapshots & Topics
Run this **last**, once every demo above has been shown, so the class sees teardown too.

In [ ]:
all_subscription_ids = [
    SUB_A_ID, SUB_B_ID, ORDERED_SUB_ID, FILTERED_SUB_ID, EXACTLY_ONCE_SUB_ID,
    DLQ_SOURCE_SUB_ID, f"{DLQ_SOURCE_SUB_ID}-dlq-check", f"training-sub-retry-{_suffix}",
    f"training-sub-push-{_suffix}",
]

for sub_id in all_subscription_ids:
    try:
        subscriber.delete_subscription(
            request={"subscription": subscriber.subscription_path(PROJECT_ID, sub_id)}
        )
        print("Deleted subscription:", sub_id)
    except gcs_exceptions.NotFound:
        pass

subscriber.delete_snapshot(request={"snapshot": snapshot_path})
print("Deleted snapshot:", SNAPSHOT_ID)

for t_path in [topic_path, schema_topic_path, dlq_topic_path]:
    publisher.delete_topic(request={"topic": t_path})
    print("Deleted topic:", t_path)

schema_client.delete_schema(request={"name": schema_path})
print("Deleted schema:", SCHEMA_ID)

In [ ]:
# Also tear down the real push-endpoint infrastructure from Section 6.2:
# the Cloud Function and its dedicated invoker service account.
try:
    region_for_cleanup = REGION if 'REGION' in dir() else 'us-central1'
    !gcloud functions delete {PUSH_FUNCTION_NAME} --gen2 --region={region_for_cleanup} \
        --project={PROJECT_ID} --quiet
    print("Deleted push Cloud Function:", PUSH_FUNCTION_NAME)
except NameError:
    print("PUSH_FUNCTION_NAME not defined in this session -- Section 6.2 may not have run.")

try:
    !gcloud iam service-accounts delete {PUSH_SA_EMAIL} --project={PROJECT_ID} --quiet
    print("Deleted push invoker service account:", PUSH_SA_EMAIL)
except NameError:
    print("PUSH_SA_EMAIL not defined in this session -- Section 6.2 may not have run.")



> 🖥️ **Check it in the console:** **Cloud Functions** (or **Cloud Run**, since gen2 functions are
> Cloud Run services underneath) → `training-push-handler-...` should no longer be listed.
> **IAM & Admin > Service Accounts** → the `training-push-invoker-...` account should also be gone.



### 15.1 Final Verification


In [ ]:

# Confirm the main topic is really gone -- this should raise NotFound if cleanup succeeded
from google.api_core import exceptions as gcs_exceptions

try:
    publisher.get_topic(request={"topic": topic_path})
    print("WARNING: main topic still exists — cleanup may not have completed.")
except gcs_exceptions.NotFound:
    print("Confirmed: main topic no longer exists.")
except NameError:
    print("topic_path not defined in this session — run the earlier cells first.")



> 🖥️ **Final console check:** **Pub/Sub > Topics**, **Pub/Sub > Subscriptions**, and
> **Pub/Sub > Snapshots** should all show no resources remaining from this notebook (`training-...`
> prefixed names) — confirming every topic, subscription, and snapshot created here was removed.
